# Reservoir Capacity Estimation — Neuer Teich

**Purpose:** Derives the elevation–area–volume (EAV) relationship for the
Neuer Teich reservoir using a power-law model, and produces the curve
table and plots needed for TALSIM-NG parameterisation.

**What it does:**
- Fits a power-law EAV curve given A_max, h_min, h_max, and V_max
- Computes the shape exponent `n` automatically from known volume at spillway
- Generates the full EAV table at user-defined elevation increments
- Plots the EAV curve in 2-D and 3-D with the full storage level (FSL) marked

**Input:** Reservoir geometry parameters (hard-coded)  
**Output:** EAV table (CSV), 2-D and 3-D plots

---

| **Parameter**                       | **Value / Info**                                  | **Remarks**                              |
| ----------------------------------- | ------------------------------------------------- | ------------------------------------------------------------ |
| **Reservoir Name**                  | TS Neuer Teich                                    |                                   |
| **Location**                        | Plothener Teiche, Saale-Orla, Thuringia           |                                |
| **Surface Area at Full Level**      | \~17.5 hectares = **175,500 m²**                  | Used in area-elevation curve                                 |
| **Max Water Elevation**             | **8.44 m** (from HWE)            | Top of storage / FSL elevation                               |
| **Min Water Elevation**             | **0.00595 m** (from HWE)             | Dead storage or lowest usable level                          | 
| **Total Volume**                    | **430,000 m³** (from the old document) | Full storage capacity                                        |
| **Low-Water Augmentation Volume**   | \~344,000 m³ (80% of total storage)               | Active storage usable for supply 0.6 l/s upto 100 days during dry periods                           |
| **Construction Year**               | 1978–1979                                         |                                  |
| **Inflows (Est. Annual)**           | \~680,000 m³/year                                 | Used for inflow time series or calibration                   |
| **Potential Water Withdrawal**      | \~228,000 m³/year (for 150 ha irrigation)         | Water use scenarios                                          |
| **Spillway / Outlet Functionality** | No current flood protection or drawdown capacity  |                       |
| **Downstream Connectivity**         | Drains into **Plothenbach** (2nd-order stream)    | For routing flows downstream (if applicaple)                                 |
| **Ownership**                       | Private ownership                                 |        |


###  k controls how quickly the reservoir area grows from bottom to top.
If elevation is below minimum: area and volume are zero.
If elevation is at or above spill elevation: area is max, volume is capped at 430,000 m³.
If k is not provided: it is estimated using the known volume at spill elevation.

In [ ]:

import numpy as np
import pandas as pd

def reservoir_capacity_powerlaw(A_max, h_min, h_max, increment, V_max):
    H = h_max - h_min
    n = (A_max * H / V_max) - 1

    data = []

    # Generate elevations from h_min to h_max **without exceeding h_max**
    elevations = list(np.arange(h_min, h_max, increment))  # note: stop at h_max (exclusive)

    # Manually add h_max if it is not in the list (to include max elevation exactly)
    if h_max not in elevations:
        elevations.append(h_max)

 
    # Sort all elevations and remove duplicates (if any)
    elevations = sorted(set(elevations))

    for h in elevations:
        if h < h_min:
            status = "Below min"
            A = 0.0
            V = 0.0
        elif abs(h - h_max) < 1e-9:
            status = "At max elevation"
            A = A_max
            V = V_max
        elif h > h_max:
            status = "Above FSL"
            A = A_max
            V = V_max
        else:
            status = "Within range"
            x = (h - h_min) / H
            A = A_max * x ** n
            V = (A_max * H / (n + 1)) * x ** (n + 1)
        
        A_km2 = A / 1e6
        V_Tsdm3 = V / 1e3
        V_acft = V * 0.000810713  # convert m³ to acre-ft

        data.append({
            "Elevation_m": round(h, 3),
            "Area_ha": round(A / 10000, 4),
            "Volume_m3": round(V, 3),
            "Volume_Tsdm3": round(V_Tsdm3, 4),
            "Volume_acft": round(V_acft, 3),
            "Status": status
        })

    return data


if __name__ == "__main__":
    print("=== Reservoir Elevation-Area-Capacity Calculator ===")
    top_area = float(input("Enter maximum reservoir surface area (m²) [default 175885.16337]: ") or 175885.16337)
    min_elevation = float(input("Enter minimum water elevation (m) [default 0.00595]: ") or 0)
    spill_elevation = float(input("Enter elevation (m) at max water capacity [default 8.44]: ") or 8.44)
    max_elevation = spill_elevation  # Force stop at 8.44 as requested
    increment = float(input("Enter elevation increment (m) [default 0.1]: ") or 0.1)
    vmax = float(input("Enter maximum volume (m³) [default 430000]: ") or 430000)

    if max_elevation <= min_elevation:
        print("Error: max elevation must be greater than min elevation.")
        exit(1)
    if increment <= 0 or increment > (max_elevation - min_elevation):
        print("Error: increment must be positive and less than elevation range.")
        exit(1)
    


    data = reservoir_capacity_powerlaw(top_area, min_elevation, max_elevation, increment, vmax)

print("\nElevation_m | Area_ha   | Volume_Tsdm³  | Status")
print("-" * 100)
for row in data:
    print(f"{row['Elevation_m']:12.3f} | {row['Area_ha']:13,.4f} | {row['Volume_Tsdm3']:14,.4f} |  {row['Status']}")
df = pd.DataFrame(data)
df.to_csv("reservoir_capacity_powerlaw_2.csv", index=False)
    #print("\nSaved to 'reservoir_capacity_powerlaw.csv'")


In [ ]:
import os
import csv
import numpy as np
import pandas as pd
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.ticker import FixedLocator, FuncFormatter
from matplotlib.patches import Patch
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt

In [ ]:

def plot_eav(df, fsl=5.5, marker_interval=1):
    plt.rcParams.update({
        'font.family':    'monospace',
        'font.size':      9,
        'axes.titlesize': 10,
        'axes.labelsize': 9,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
    })
    
    # === Filter and prepare data ===
    df_line = df[df["Elevation_m"] <= fsl].copy()
    df_line["Volume_Tsdm3"] = df_line["Volume_Tsdm3"].replace(0, np.nan)

    elevation = df_line["Elevation_m"].values
    volume = df_line["Volume_Tsdm3"].values
    area = df_line["Area_ha"].values

    valid_vol = volume[~np.isnan(volume)]
    valid_area = area[~np.isnan(area)]
    max_vol = valid_vol.max() if valid_vol.size else 0
    max_area = valid_area.max() if valid_area.size else 0

    # === Create figure ===
    fig, ax_vol = plt.subplots(figsize=(6.5, 4.5)) 

    mark_indices = list(range(0, len(elevation), marker_interval))

    # --- Bottom axis: Capacity (Volume) vs Elevation ---
    ax_vol.plot(volume, elevation,
                color='blue', lw=1,
                marker='D', markersize=4, markevery=mark_indices,
                label="Volume")

    ax_vol.set_xlabel("(Capacity / Volume Tsd m³)", color='blue')
    ax_vol.set_ylabel("Elevation (m)", color='black')
    ax_vol.tick_params(axis='x', labelcolor='blue')
    ax_vol.tick_params(axis='y', labelcolor='black')
    ax_vol.set_ylim(0, 6)

    # --- Top axis: Area vs Elevation ---
    ax_area = ax_vol.twiny()
    ax_area.plot(area, elevation,
                 color='brown', lw=1, linestyle='--',
                 marker='^', markersize=4, markevery=mark_indices,
                 label="Area")

    ax_area.set_xlabel("Area (ha)", color='brown')
    ax_area.tick_params(axis='x', labelcolor='brown')

    # Match elevation (y) limits and ticks
    ax_area.set_ylim(ax_vol.get_ylim())
    ax_area.set_yticks(ax_vol.get_yticks())
    ax_area.tick_params(axis='y', labelcolor='black')

    # === Grid and tick spacing ===
    ax_vol.xaxis.set_major_locator(FixedLocator(np.arange(0, max_vol + 50, 50)))
    ax_area.xaxis.set_major_locator(FixedLocator(np.arange(0, max_area + 2, 2)))
    ax_vol.yaxis.set_major_locator(FixedLocator(np.arange(0, 6, 0.5)))

    # === FSL line ===
    ax_vol.axhline(fsl, color='red', linestyle=':', lw=1.5)
    ax_vol.text(0, fsl + 0.15, f"Maximum Capacity = 300.000 Tsd m3 m", color='red', va='bottom',fontfamily='monospace')

    # === ASL line (80% of FSL volume) ===
    target_volume = 213.00  # 80% of 430,000 m³
    df_valid = df_line.dropna(subset=["Volume_Tsdm3"])
    asl_row = df_valid.iloc[(df_valid["Volume_Tsdm3"] - target_volume).abs().argsort()[:1]]
    asl_elev = asl_row["Elevation_m"].values[0]

    ax_vol.axhline(asl_elev, color='darkgreen', linestyle='--', lw=1.5)
    ax_vol.text(0, asl_elev + 0.15, f"Spillway ≈  213.000 Tsd m3",
                color='darkgreen', va='bottom',fontfamily='monospace')

    # === Legends ===
    ax_vol.legend(loc='lower right')
    ax_area.legend(loc='upper right')

    # === Title and note ===
    #plt.title("TS_Neuer Teich: Elevation–Area–Capacity Curve")

    ax_vol.grid(True, linestyle='--', alpha=0.5)
    # === Fix x-axis alignment ===
    ax_vol.set_xlim(0, max_vol)
    ax_area.set_xlim(0, max_area)
    plt.tight_layout()
    plt.savefig("eav_curve.png", dpi=300, bbox_inches='tight')
    plt.show()
df = pd.DataFrame(data)
plot_eav(df, marker_interval=5)


In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib import ticker
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [ ]:
# --- Load CSV ---
csv_path = r"C:\Users\raah\Desktop\Analysis_raah\raah\Scripts\reservoir_capacity_powerlaw_2.csv"
reservoir_data = []
with open(csv_path, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        reservoir_data.append({
            "Elevation_m": float(row["Elevation_m"]),
            "Area_m2": float(row["Area_m2"]),
            "Volume_M3": float(row["Volume_m3"])
        })

# --- Spillway Info ---
max_elevation = 8.44  # meters

# --- Extract and Process Data ---
elevs = [round(row["Elevation_m"], 2) for row in reservoir_data]
areas = [row["Area_m2"] for row in reservoir_data]
volumes = [row["Volume_M3"] for row in reservoir_data]

min_elevation = min(elevs)
depths = [e - min_elevation for e in elevs]

max_depth = max_elevation - min_elevation
max_volume = np.interp(max_depth, depths, volumes)

max_volume = max(volumes)

# --- User input: HWE water elevation (absolute height) ---
try:
    user_input = input(f"Enter spillway water elevation (m) [min = {min_elevation}, max = {max(elevs)}]: ")
    target_elevation = float(user_input) if user_input.strip() else max(elevs)
except ValueError:
    print("Invalid input. Please enter a numeric value.")
    exit()

# Check bounds
if target_elevation < min_elevation or target_elevation > max(elevs):
    print(f"Elevation out of range ({min_elevation} - {max(elevs)} m)")
    exit()

# Convert elevation → depth
target_depth = target_elevation - min_elevation

# Convert depth → volume
target_volume = np.interp(target_depth, depths, volumes)

# This depth is the fill depth used in plotting
fill_depth = target_depth

# --- Sort data ---
sorted_data = sorted(zip(elevs, depths, areas, volumes), key=lambda x: x[0])
elevs, depths, areas, volumes = zip(*sorted_data)

fig = plt.figure(figsize=(16, 8))
ax = fig.add_subplot(121, projection='3d')

def build_layer_verts(d1, d2, a1, a2):
    w1 = np.sqrt(a1)
    w2 = np.sqrt(a2)

    x1 = np.array([-w1/2, w1/2, w1/2, -w1/2])
    y1 = np.array([-w1/2, -w1/2, w1/2, w1/2])
    z1 = np.full(4, d1)

    x2 = np.array([-w2/2, w2/2, w2/2, -w2/2])
    y2 = np.array([-w2/2, -w2/2, w2/2, w2/2])
    z2 = np.full(4, d2)

    verts = [
        list(zip(x1, y1, z1)),  # bottom
        list(zip(x2, y2, z2)),  # top
        list(zip([x1[0], x1[1], x2[1], x2[0]],
                 [y1[0], y1[1], y2[1], y2[0]],
                 [z1[0], z1[1], z2[1], z2[0]])),
        list(zip([x1[1], x1[2], x2[2], x2[1]],
                 [y1[1], y1[2], y2[2], y2[1]],
                 [z1[1], z1[2], z2[2], z2[1]])),
        list(zip([x1[2], x1[3], x2[3], x2[2]],
                 [y1[2], y1[3], y2[3], y2[2]],
                 [z1[2], z1[3], z2[3], z2[2]])),
        list(zip([x1[3], x1[0], x2[0], x2[3]],
                 [y1[3], y1[0], y2[0], y2[3]],
                 [z1[3], z1[0], z2[0], z2[3]]))
    ]
    return verts

# --- Plot filled volume layers ---
for i in range(len(depths) - 1):
    d1, d2 = depths[i], depths[i + 1]
    a1, a2 = areas[i], areas[i + 1]

    # Skip layers fully above fill depth
    if d1 >= fill_depth:
        continue

    layer_top = min(d2, fill_depth)
    verts = build_layer_verts(d1, layer_top, a1, a2)

    # Color based on spillway crossing
    if target_volume > spill_volume and d1 >= spill_depth:
        color = 'red'
    else:
        color = 'cyan'

    poly = Poly3DCollection(verts, facecolors=color, edgecolors='grey', linewidths=0.2, alpha=0.6)
    ax.add_collection3d(poly)

# --- Plot empty (hollow) volume layers if any ---
if target_volume < spill_volume:
    for i in range(len(depths) - 1):
        d1, d2 = depths[i], depths[i + 1]
        a1, a2 = areas[i], areas[i + 1]

        # Only layers strictly above fill_depth and up to spill_depth
        if d2 <= fill_depth or d1 >= spill_depth:
            continue

        layer_bottom = max(d1, fill_depth)
        layer_top = min(d2, spill_depth)

        verts = build_layer_verts(layer_bottom, layer_top, a1, a2)

        poly_empty = Poly3DCollection(verts, facecolors=(1,1,1,0), edgecolors='grey', linewidths=0.5, alpha=0)
        ax.add_collection3d(poly_empty)

ax.set_title(f"Reservoir Volume: {target_volume:.2f} m³\n(Fill Depth ≈ {fill_depth:.2f} m)", fontsize=12)
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Depth (m)")
ax.set_zlim(0, max(depths) * 1.1)
ax.view_init(elev=30, azim=45)

legend_elements = [
    Patch(facecolor='cyan', label='Filled (≤ Spillway)'),
    Patch(facecolor='red', label='Filled Above Spillway'),
    Patch(facecolor=(1,1,1,0), edgecolor='lightgrey', label='Remaining Capacity')
]
ax.legend(handles=legend_elements, loc='upper left')

# --- 2D plot: Depth vs Volume ---
volume_ax = fig.add_subplot(122)

# Plot volume profile
volume_ax.plot(depths, volumes, 'go-', label='Volume Profile')

if np.isclose(target_volume, spill_volume):
    volume_ax.axhline(spill_volume, color='purple', linestyle='--', label=f'Maximum Capacity ({spill_volume:.2f} m³)')
else:
    volume_ax.axhline(target_volume, color='blue', linestyle='--', label=f'Spillway/HWE ({target_volume:.2f} m³)')
    volume_ax.axhline(spill_volume, color='purple', linestyle='--', label=f'Maximum Reservoir Capacity ({spill_volume:.2f} m³)')

# --- Set custom ticks ---
max_volume = max(volumes)
max_depth = max(depths)

volume_ax.set_yticks(np.arange(0, max_volume + 25000, 25000))  # Y-axis every 5000
volume_ax.set_xticks(np.arange(0, max_depth + 0.5, 0.5))     # X-axis every 0.5

volume_ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
volume_ax.set_xlabel("Depth from Base (m)")
volume_ax.set_ylabel("Volume (m³)")
volume_ax.set_title("Volume vs. Depth")
volume_ax.grid(True)
volume_ax.legend()


plt.tight_layout()
plt.show()
